# Patches — Bloc 3c, Sessió 5
### Efectes en temps real: teoria, visualització i ipywidgets

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

⚠️ **Nota:** El codi de so en temps real (`sd.Stream`) s'executa a **Thonny** (`exemples.py`). Aquest notebook conté la teoria, la visualització del buffer de retard i els sliders amb `ipywidgets` per entendre els paràmetres **sense necessitar so real**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sample_rate = 44100

## 1. El buffer de retard — simulació pas a pas

Simulem 5 crides al callback d'eco per veure com evoluciona el buffer.

In [ ]:
# Parametres de la simulació
buffer_size = 6   # buffer petit per visualitzar
frames = 2        # mostres per crida (molt petit per il·lustrar)
decay = 0.5

# Simulem 4 blocs d'entrada
blocs_entrada = [
    np.array([0.8, 0.6]),
    np.array([0.4, 0.2]),
    np.array([0.9, 0.1]),
    np.array([0.3, 0.7]),
]

buffer = np.zeros(buffer_size)

print(f"{'Crida':>6} {'Entrada':>16} {'Eco llegit':>16} {'Sortida':>16} {'Buffer després':>25}")
print("-" * 85)

for i, bloc in enumerate(blocs_entrada):
    eco = buffer[-frames:]
    sortida = bloc + eco * decay
    buffer = np.roll(buffer, -frames)
    buffer[-frames:] = bloc
    print(f"{i+1:>6} {str(np.round(bloc,2)):>16} {str(np.round(eco,2)):>16} "
          f"{str(np.round(sortida,2)):>16} {str(np.round(buffer,2)):>25}")

## 2. Simulació completa: eco offline per entendre el buffer

Simulem el mateix callback sobre un senyal sintètic i escoltem el resultat. Amb Colab podem sentir el resultat de la simulació (no el temps real).

In [ ]:
def simula_eco_callback(signal, delay_seconds, decay, blocksize=512, sample_rate=44100):
    """Simula el comportament del callback d'eco bloc a bloc."""
    delay_samples = int(delay_seconds * sample_rate)
    delay_buffer = np.zeros(delay_samples, dtype='float32')
    output = np.zeros_like(signal)

    for i in range(0, len(signal), blocksize):
        bloc = signal[i:i+blocksize]
        frames = len(bloc)

        eco = delay_buffer[:frames]  # llegim les mes antigues
        out = bloc + eco * decay

        delay_buffer = np.roll(delay_buffer, -frames)
        delay_buffer[-frames:] = bloc

        output[i:i+frames] = out

    return output

# Generem un senyal de prova: uns quants impulsos
duration = 2.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
signal = np.zeros_like(t, dtype='float32')
for pos in [0.2, 0.6, 1.0, 1.4]:
    idx = int(pos * sample_rate)
    signal[idx:idx+int(0.05*sample_rate)] = 0.8 * np.sin(
        2*np.pi*440*t[:int(0.05*sample_rate)])

# Apliquem eco
resultat = simula_eco_callback(signal, delay_seconds=0.3, decay=0.5)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5))
ax1.plot(t, signal)
ax1.set_title('Senyal original')
ax2.plot(t, resultat)
ax2.set_title('Amb eco (delay=0.3s, decay=0.5)')
plt.tight_layout()
plt.show()

print("Senyal original:")
display(Audio(signal, rate=sample_rate))
print("Amb eco:")
display(Audio(resultat, rate=sample_rate))

## 3. Control amb ipywidgets — slider interactiu

Explorem com canvien els paràmetres de l'eco amb sliders. Això no és so en temps real — és una demo visual/auditiva per entendre els paràmetres **abans** d'anar a Thonny.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

out = widgets.Output()

def actualitza(delay_ms, decay, drive):
    with out:
        clear_output(wait=True)

        # Apliquem distorsió primer, eco després
        dist = np.clip(signal * drive, -0.8, 0.8) / 0.8
        resultat = simula_eco_callback(dist, delay_seconds=delay_ms/1000,
                                        decay=decay, sample_rate=sample_rate)

        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(t, resultat, alpha=0.8)
        ax.set_title(f'delay={delay_ms}ms, decay={decay:.2f}, drive={drive:.1f}')
        ax.set_ylim(-1.5, 1.5)
        plt.tight_layout()
        plt.show()

        display(Audio(np.clip(resultat, -1, 1), rate=sample_rate))

controls = widgets.interactive(
    actualitza,
    delay_ms=widgets.IntSlider(min=50, max=600, step=10, value=300,
                                description='Delay (ms)'),
    decay=widgets.FloatSlider(min=0.0, max=0.95, step=0.05, value=0.5,
                               description='Decay'),
    drive=widgets.FloatSlider(min=1.0, max=8.0, step=0.5, value=1.0,
                               description='Drive'),
)

display(controls, out)

## 4. Per qué les classes seran millors — visualització del problema

Dos ecos simultanis amb l'enfocament de buffer global:

In [ ]:
# Amb buffer global: necessitem dues variables separades
delay_buffer_1 = np.zeros(int(0.2 * sample_rate))  # eco curt
delay_buffer_2 = np.zeros(int(0.5 * sample_rate))  # eco llarg
# ...i el callback hauria de gestionar totes dues manualment
# Si en volem 5 ecos? 5 variables globals...

print("Amb buffer global (actual):")
print("  delay_buffer_1 = np.zeros(...)  # eco 1")
print("  delay_buffer_2 = np.zeros(...)  # eco 2")
print("  delay_buffer_3 = np.zeros(...)  # eco 3 ...")
print()
print("Amb classes (Bloc 5):")
print("  eco1 = EcoEffect(delay=0.2, decay=0.5)")
print("  eco2 = EcoEffect(delay=0.5, decay=0.3)")
print("  eco3 = EcoEffect(delay=0.8, decay=0.2)")
print("  # Cada instància porta el seu propi buffer intern")
print()

# Simulació amb dos ecos
eco1 = simula_eco_callback(signal, delay_seconds=0.2, decay=0.4)
eco2 = simula_eco_callback(signal, delay_seconds=0.5, decay=0.3)
dos_ecos = np.clip(eco1 + eco2, -1, 1)

print("Resultat: dos ecos simultanis (0.2s + 0.5s):")
display(Audio(dos_ecos, rate=sample_rate))